### RecursiveCharacterTextSplitter

- Document를 설정한 chunk를 기준으로 분리


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content="""
        환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.
        """
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"{i} : {chunk.page_content}")

0 : 환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.


### Vector DB

- TextSplitter로 분리한 데이터를 Embedding 처리한 후 저장


In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

embedding = OpenAIEmbeddings(model="text-embedding-3-small")

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("documents : ", len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

print("chunk : ", len(chunks))

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

print("FIASS 생성완료")

query = "펀드가 무엇인가요?"

result = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(result):
    print("결과 : ", i)
    print("페이지 : ", doc.metadata.get("page"))
    print()
    print(doc.page_content)
    print()

/var/folders/qz/hyvszw9n4ms9wd1s5wn7fm2c0000gn/T/ipykernel_4176/2439791718.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


documents :  191
chunk :  332
FIASS 생성완료
결과 :  0
페이지 :  28

으로 지급받도록 되어 있으므로, 펀드의 수익률에 큰 영향을 미치게 됩니다. 따라서 펀드 선택시 보수와 
수수료가 어떻게 되는지도 살펴보는 것이 좋습니다.
1) 펀드 투자요령
Q.
A.
펀드 가입시 고려해야 할 사항에는 어떤 것들이 있습니까?
펀드는 원금손실가능성이 있는 실적배당상품으로 가입시 고려하여야 할 기본적인 사항들을 예를 들어
보면 다음과 같습니다.
① 투자기간, 투자유형, 투자성향 등 투자목적에 맞는 펀드를 선택해야 합니다.
② 펀드가입시기 결정 등과 관련하여 주식시장 및 채권시장의 흐름을 면밀히 살펴야 합니다.
③  운용회사나 펀드매니저에 따라 운용성과에 차이가 많이 발생할 수 있으므로 운용회사의 과거 실적 
및 운용전략, 펀드매니저의 운용성향 등도 사전에 확인하여야 합니다.
④ 갑작스런 자금수요에 대비하여 사전에 중도환매가 가능한지를 꼭 확인하여야 합니다.
⑤ 판매담당인력의 전문성 및 판매상품의 다양성 등을 고려하여 판매회사를 골라야 합니다.

결과 :  1
페이지 :  31

것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있

### RecursiveCharacterTextSplitter 및 Vector DB로 LLM까지 연결


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)

embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]{question}",
        ),
    ]
)

parser = StrOutputParser()

# question = "펀드란 무엇인가요?"
question = "대한민국의 수도는 어디인가요?"

retriever_docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in retriever_docs])

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result : ", result)

result :  제공된 문서에서 답을 찾을 수 없습니다.


### Vector DB에서 similarity_search_with_score로 통한 유사도 점수 확인


In [ ]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

questions = [
    "펀드란 무엇인가요?",
    "대한민국의 수도는 어디인가요?",
    "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"
]

for question in questions:
    result = vectorstore.similarity_search_with_score(question, k=3)

    print(f"질문: {question}")

    for i, (doc, score) in enumerate(result):
        print("-" * 100)
        print("검색 결과")
        print("distance: ", score)
        print("page: ", doc.metadata.get("page"))
        print("내용: ", doc.page_content[:500])

질문 :  펀드란 무엇인가요?
검색 결과
distance :  1.0027242
page :  28
내용 :  으로 지급받도록 되어 있으므로, 펀드의 수익률에 큰 영향을 미치게 됩니다. 따라서 펀드 선택시 보수와 
수수료가 어떻게 되는지도 살펴보는 것이 좋습니다.
1) 펀드 투자요령
Q.
A.
펀드 가입시 고려해야 할 사항에는 어떤 것들이 있습니까?
펀드는 원금손실가능성이 있는 실적배당상품으로 가입시 고려하여야 할 기본적인 사항들을 예를 들어
보면 다음과 같습니다.
① 투자기간, 투자유형, 투자성향 등 투자목적에 맞는 펀드를 선택해야 합니다.
② 펀드가입시기 결정 등과 관련하여 주식시장 및 채권시장의 흐름을 면밀히 살펴야 합니다.
③  운용회사나 펀드매니저에 따라 운용성과에 차이가 많이 발생할 수 있으므로 운용회사의 과거 실적 
및 운용전략, 펀드매니저의 운용성향 등도 사전에 확인하여야 합니다.
④ 갑작스런 자금수요에 대비하여 사전에 중도환매가 가능한지를 꼭 확인하여야 합니다.
⑤ 판매담당인력의 전문성 및 판매상품의 다양성 등을 고려하여 판매회사를 골라야 합니다.
검색 결과
distance :  1.0495431
page :  31
내용 :  것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 

### MMR(Maximum Marginal Relevance)

- 질문과 관련 있으면서 서로 다른 정보를 가진 문서를 검색


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

question = "펀드의 종류와 특징은 무엇인가요?"

similarity_retriever = vectorstore.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 3}
)
# print(similarity_retriever)

similarity_docs = similarity_retriever.invoke(question)
# print(similarity_docs) # document형태

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3, # 서로 너무 비슷하지 않도록 고려해서 최종적으로 3개를 골라
        "fetch_k": 10, # MMR이 최종 3개를 고르기 전에 먼저 살펴볼 후보의 수
        "lambda_mult": 0.5, # 관련성과 다양성은 50:50으로 봐 (MMR의 진짜 핵심 옵션)
                            # 1.0	관련성 중심 / 0.5	관련성 + 다양성 균형 / 0.0	다양성 중심
    }
)
mmr_docs = mmr_retriever.invoke(question)

print("\n")
print("=" * 80)
print("Similarity Search 결과")
print("=" * 80)

for i, doc in enumerate(similarity_docs):
    print(f"\n----- 결과 {i + 1} -----")
    print("Page: ", doc.metadata.get("page"))
    print(doc.page_content[:500])


# ============================================================
# MMR 결과
# ============================================================
print("\n")
print("=" * 80)
print("MMR Search 결과")
print("=" * 80)

for i, doc in enumerate(mmr_docs):
    print(f"\n----- 결과 {i + 1} -----")
    print("Page: ", doc.metadata.get("page"))
    print(doc.page_content[:500])



Similarity Search 결과

----- 결과 1 -----
Page:  28
으로 지급받도록 되어 있으므로, 펀드의 수익률에 큰 영향을 미치게 됩니다. 따라서 펀드 선택시 보수와 
수수료가 어떻게 되는지도 살펴보는 것이 좋습니다.
1) 펀드 투자요령
Q.
A.
펀드 가입시 고려해야 할 사항에는 어떤 것들이 있습니까?
펀드는 원금손실가능성이 있는 실적배당상품으로 가입시 고려하여야 할 기본적인 사항들을 예를 들어
보면 다음과 같습니다.
① 투자기간, 투자유형, 투자성향 등 투자목적에 맞는 펀드를 선택해야 합니다.
② 펀드가입시기 결정 등과 관련하여 주식시장 및 채권시장의 흐름을 면밀히 살펴야 합니다.
③  운용회사나 펀드매니저에 따라 운용성과에 차이가 많이 발생할 수 있으므로 운용회사의 과거 실적 
및 운용전략, 펀드매니저의 운용성향 등도 사전에 확인하여야 합니다.
④ 갑작스런 자금수요에 대비하여 사전에 중도환매가 가능한지를 꼭 확인하여야 합니다.
⑤ 판매담당인력의 전문성 및 판매상품의 다양성 등을 고려하여 판매회사를 골라야 합니다.

----- 결과 2 -----
Page:  30
사후관리 서비스를 제공합니다. 펀드판매사가 제공하는 보고서에는 분기별 자산운용보고서, 투자자별 펀드잔고 
통보서, 자산보관·관리보고서 등이 있습니다. 
•자산운용보고서 확인 : 자산운용사가 작성하여 판매회사가 송부하는 분기별 자산운용보고서를 활용하여 내가
가입한 펀드가 어떻게 운용되고 있는지를 확인합니다. 자산운용보고서를 통해 운용성과 개요 및 운용 기간 중 
손익, 펀드자산내역, 매매금액 및 매매 회전율, 향후 운용계획 등의 정보가 제공됩니다. 
•‘투자자별 펀드잔고 통보’서비스 활용 : 판매사는 ‘투자자별 펀드잔고 통보’ 서비스를 제공하는데, 이 서비스는
투자자의 펀드잔고, 개별 수익률 정보 등을 전자우편 등을 통해 제공하는 서비스로 모든 투자자에게 제공됩니다. 
다만, 제공을 원하지 않을 경우 별도의 신청을 통하여 제공 받지 않을 수 있습니다.
▒ 적립식/거치식 펀

### Metadata Filtering

- Vector DB에서 검색하기 전에 특정 조건에 맞는 문서만 검색 대상으로 제한


In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)
vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

question = "펀드란 무엇인가?"

all_results = vectorstore.similarity_search(
    question,
    k=3
)
# print(all_results)

for i, doc in enumerate(all_results):
    print(f"\n----- 결과 {i + 1} -----")
    print("[Page] ", doc.metadata.get("page"))
    print("[Content]")
    print(doc.page_content[:500])

filtered_results = vectorstore.similarity_search(
    question,
    k=3,
    filter={"page": 50} # metadata의 page가 50인 것만 검색 대상으로 해라
)

print()

if not filtered_results:
    print("조건에 맞는 문서를 찾지 못했습니다.")
else:
    for i, doc in enumerate(filtered_results):
        print(f"\n----- 결과 {i + 1} -----")
        print("[Page]", doc.metadata.get("page"))
        print("[Content]")
        print(doc.page_content[:500])


----- 결과 1 -----
[Page]  28
[Content]
으로 지급받도록 되어 있으므로, 펀드의 수익률에 큰 영향을 미치게 됩니다. 따라서 펀드 선택시 보수와 
수수료가 어떻게 되는지도 살펴보는 것이 좋습니다.
1) 펀드 투자요령
Q.
A.
펀드 가입시 고려해야 할 사항에는 어떤 것들이 있습니까?
펀드는 원금손실가능성이 있는 실적배당상품으로 가입시 고려하여야 할 기본적인 사항들을 예를 들어
보면 다음과 같습니다.
① 투자기간, 투자유형, 투자성향 등 투자목적에 맞는 펀드를 선택해야 합니다.
② 펀드가입시기 결정 등과 관련하여 주식시장 및 채권시장의 흐름을 면밀히 살펴야 합니다.
③  운용회사나 펀드매니저에 따라 운용성과에 차이가 많이 발생할 수 있으므로 운용회사의 과거 실적 
및 운용전략, 펀드매니저의 운용성향 등도 사전에 확인하여야 합니다.
④ 갑작스런 자금수요에 대비하여 사전에 중도환매가 가능한지를 꼭 확인하여야 합니다.
⑤ 판매담당인력의 전문성 및 판매상품의 다양성 등을 고려하여 판매회사를 골라야 합니다.

----- 결과 2 -----
[Page]  31
[Content]
것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니

### Hybrid Search

- Vector Search(의미가 비슷한 문서 반환)과 Keyword Search(정확한 단어로 검색)를 모두 활용


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

model = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
bm25_retriever = BM25Retriever.from_documents(chunks, k=3)

hybrid_retriever = EnsembleRetriever(
    retrievers = [vector_retriever, bm25_retriever],
    weight = [0.5, 0.5]
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

retrieved_docs = hybrid_retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)
parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

모바일폰을 사용한 금융거래 시 유의해야 할 점은 다음과 같습니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기.
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않기.
3. 금융거래 비밀번호를 안전하게 관리하기.
4. 모바일폰 분실·도난 시 모바일폰 금융서비스 사용 중지하기.
5. 모바일폰 교체·수리 전 중요 정보 삭제하기.
6. 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기.
7. 모바일폰 사용환경을 임의로 변경하지 않기.
8. 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기.
9. 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기.
10. 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용 시 주의하기.


### Reranker

- 찾아온 문서들 중에서 질문과 관련 있는 순서대로 다시 정리
- 일반적으로 Retriever보다 계산량이 많기에 후보를 추려오면 그 뒤 사용


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

model = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",   # 다국어 지원, 한국어 OK
    model_kwargs={"device": "mps"},         # Mac GPU 사용 (Windows 강의의 "cuda" 자리)
)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

retrieved_docs = hybrid_retriever.invoke(question)
print("검색된 후보 문서 : ", len(retrieved_docs))

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

final_docs = reranked_docs   # top_n=3에서 이미 잘려서 나옴

for i, doc in enumerate(final_docs, start=1):
    print(f"\n--- 최종 문서 {i} ---")
    print("[Page]", doc.metadata.get("page"))
    print(doc.page_content[:500])

context = "\n\n".join([doc.page_content for doc in final_docs])

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]\n{question}",
        ),
    ]
)

/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_25966/2685781352.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


검색된 후보 문서 :  18

--- 최종 문서 1 ---
page: 25
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
 금융투자상품에 투자하기   25
① 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기 
② 모바일폰이나 인터넷에 금융정보를 저장하지 않기
③ 금융거래 비밀번호를 안전하게 관리하기
④ 모바일폰 분실·도난시 모바일폰 금융서비스 사용 중지하기.
⑤ 모바일폰 교체·수리 전 중요 정보 삭제하기
⑥ 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기
⑦ 모바일폰 사용환경을 임의로 변경하지 않기
⑧ 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기
⑨ 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기
⑩ 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용시 주의하기
모바일폰 금융거래 10계명
모바일폰은 언제, 어디에서나 사용하는 필수품이 되었습니다. 이제는 금융거래도 모바일폰을 이용하여 언제, 
어디서든지 할 수 있습니다.  
여러 금융기관에서는 금리

--- 최종 문서 2 ---
page: 16
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집16
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
나날이 복잡해져 가는 금융투자상품에 대해 정확
하게 이해하기 위하여 투자설명서를 꼼꼼히 체크
하여야 하며, 이해가 가지 않는 부분이 있다면 금
융투자회사 직원에게 설명을 요청하는 것이 좋습
니다.
특히, 계좌개설 시 반드시 투자설명서 및 약관을 
교부받아 적어도 투자기간 동안 보관해 두어야 합
니다.
금융투자상품 가입 시 작성하는 서류에 투자자 본
인이 자필로 서명하거나 기재하는 문구가 여러가
지 있습니다. 특히 투자설명서의 이해 여부를 확인하기 위한 본인기재사항은 투자자가 해당 금융투자상품을 
이해하고 투자한다는 것을 선언하는 것으로 사후 법적다툼이 발생할 때 판단하는 1차적인 자료로 활용될 수 
있으므로 그 내용을 충분히 확인한 후 작성하는 

### Multi-Query

- 하나의 질문을 여러 개의 검색 질문으로 변환한 뒤 각각 검색하는 방법
- 벡터 검색은 의미가 비슷하면 어느 정도 찾아주지만, 한 번의 검색으로는 놓치는 문서가 생길 수 있습니다.


In [3]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

question = "모바일 폰을 사용한 금융 거래 시 어떤 점을 유의해야 하는가?"

prompt = ChatPromptTemplate.from_template("""
    다음 질문에 대해 문서 검색에 사용할 수 있는 서로 다른 검색 질문 3개를 만들어주세요.

    원래 질문
    {question}

    조건:
    - 원래 질문의 의미는 유지할 것
    - 서로 다른 표현을 사용할 것
    - 검색에 도움이 되는 핵심 단어를 포함할 것
    - 번호나 설명없이 검색 질문만 한 줄씩 출력할 것
""")

parser = StrOutputParser()

cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",   # 다국어 지원, 한국어 OK
    model_kwargs={"device": "mps"},         # Mac GPU 사용 (Windows 강의의 "cuda" 자리)
)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

chain = prompt | llm | parser

generated_queries_text = chain.invoke({"question": question})
generated_queries = [
    query.strip() for query in generated_queries_text.split("\n") if query.strip()
]

all_docs = []

for i, query in enumerate(generated_queries, start=1):
    print(query)
    docs = hybrid_retriever.invoke(query)

    print("검색된 문서 수:", len(docs))
    all_docs.extend(docs) # 검색 결과를 하나의 리스트에 모읍니다.

print("\n전체 검색 결과:", len(all_docs))

unique_docs = {}

for doc in all_docs:
    # page = doc.metadata.get("page")

    # if page not in unique_docs:
    #     unique_docs[page] = doc
    key = doc.page_content
    if key not in unique_docs:
        unique_docs[key] = doc

unique_docs = list(unique_docs.values())
print("중복 제거 후 문서 수 : ", len(unique_docs))

reranked_docs = reranker.compress_documents(documents=unique_docs, query=question)

final_docs = reranked_docs

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result: ", result)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 11165.04it/s]


모바일 기기로 금융 거래를 할 때 주의할 점은 무엇인가요?
검색된 문서 수: 19
스마트폰으로 금융 거래 시 고려해야 할 사항은 어떤 것들이 있나요?
검색된 문서 수: 19
휴대폰을 이용한 금융 거래에서 유의해야 할 위험 요소는 무엇인가요?
검색된 문서 수: 18

전체 검색 결과: 56
중복 제거 후 문서 수 :  40
result:  모바일 폰을 사용한 금융 거래 시 유의해야 할 점은 다음과 같습니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스를 이용해야 합니다.
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않아야 합니다.
3. 금융거래 비밀번호를 안전하게 관리해야 합니다.
4. 모바일폰 분실·도난 시 모바일폰 금융서비스 사용을 중지해야 합니다.
5. 모바일폰 교체·수리 전 중요 정보를 삭제해야 합니다.
6. 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP)를 이용해야 합니다.
7. 모바일폰 사용환경을 임의로 변경하지 않아야 합니다.
8. 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사를 해야 합니다.
9. 모바일폰 '잠금기능'을 설정하고 '잠금비밀번호'는 수시로 변경해야 합니다.
10. 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용 시 주의해야 합니다.


### Query Transformation

- 사용자 질문을 그대로 검색하지 않고 검색하기 좋은 형태로 바꿔주는 것
- 1. Rewrite, Step-back, 3. HyDE 등


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

question = "그거 모바일로 금융거래 할 때 조심해야 하는 게 뭐였지?"

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로 다시 작성하세요.

    조건:
    - 질문의 원래 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()

rewrite_chain = rewrite_prompt | llm | parser
rewritten_question = rewrite_chain.invoke({"question": question})

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

final_docs = retrieved_docs[:3]

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

모바일로 금융거래 할 때 조심해야 하는 사항은 다음과 같습니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않기
3. 금융거래 비밀번호를 안전하게 관리하기
4. 모바일폰 분실·도난시 모바일폰 금융서비스 사용 중지하기
5. 모바일폰 교체·수리 전 중요 정보 삭제하기
6. 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기
7. 모바일폰 사용환경을 임의로 변경하지 않기
8. 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기
9. 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기
10. 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용시 주의하기


### Context Compression

- 검색된 문서에서 질문에 필요한 부분만 뽑아내는 것


In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",
    model_kwargs={"device": "mps"},    
)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=5)

question = "모바일 폰을 사용한 금융 거래 시 어떤 점을 유의해야 하는가?"

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로 다시 작성하세요.

    조건:
    - 원래 질문의 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()

rewrite_chain = rewrite_prompt | llm | parser
rewritten_question = rewrite_chain.invoke({"question": question})
print("재작성된 질문:", rewritten_question)

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

compressor = LLMChainExtractor.from_llm(llm)

# 압축 반복문 필요업슴 -> compress_documents는 원래 문서 목록을 받는 메서드라서.
# compressed_docs = []
# for doc in reranked_docs:
#     compressed_doc = compressor.compress_documents(documents=[doc], query=question)

#     compressed_docs.extend(compressed_doc)

# final_docs = compressed_docs[:5]
final_docs = compressor.compress_documents(documents=reranked_docs, query=question)
print("압축 후 문서 수:", len(final_docs))

context = "\n\n".join(doc.page_content for doc in final_docs)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10842.91it/s]


재작성된 질문: 모바일 폰 금융 거래 시 유의사항은 무엇인가?
압축 후 문서 수: 4
모바일 폰을 사용한 금융 거래 시 다음의 사항에 유의해야 합니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스를 이용합니다.
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않습니다.
3. 금융거래 비밀번호를 안전하게 관리합니다.
4. 모바일폰 분실·도난 시에는 모바일폰 금융서비스 사용을 중지합니다.
5. 모바일폰 교체·수리 전에는 중요 정보를 삭제합니다.
6. 휴대폰 문자서비스(SMS) 및 일회용비밀번호(OTP)를 이용합니다.
7. 모바일폰 사용환경을 임의로 변경하지 않습니다.
8. 모바일폰 보안 업데이트를 정기적으로 수행하고 바이러스 검사를 합니다.
9. 모바일폰의 ‘잠금기능’을 설정하고 ‘잠금비밀번호’를 수시로 변경합니다.
10. 출처가 불분명하거나 보안 설정 없는 무선랜(Wi-Fi) 사용 시 주의합니다.

특히 금융거래를 위해서는 모바일폰의 보안이 무엇보다 중요하므로, 공공장소에서 비밀번호가 없는 Wi-Fi를 사용할 경우 해킹의 위험에 노출될 수 있으니 특별히 주의해야 합니다.


### Citation

- 답변의 근거가 어디인지 같이 보여주는 것


In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로
    다시 작성하세요.

    조건:
    - 질문의 원래 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()



rewrite_chain = rewrite_prompt | llm | parser
rewritten_question = rewrite_chain.invoke({"question": question})
print("Rewrite:")
print(rewritten_question)

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",   # 다국어 지원, 한국어 OK
    model_kwargs={"device": "mps"},         # Mac GPU 사용 (Windows 강의의 "cuda" 자리)
)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)
reranker_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

context_parts = []

for doc in reranked_docs:
    page = doc.metadata.get("page")
    page = page + 1 if isinstance(page, int) else "unknown"
    source = os.path.basename(doc.metadata.get("source", "unkown"))
    context_parts.append(
        f"[출처: p.{page}]\n[파일: {source}]\n{doc.page_content}"
    )


context = "\n\n".join(context_parts)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로
            질문에 답변하는 도우미입니다.

            반드시 제공된 Context만 근거로 답변하세요.

            답변에 사용한 문서의 출처를
            반드시 다음 형식으로 표시하세요.

            [출처: p.페이지번호]

            Context에 없는 내용은 추측하지 마세요.

            문서에서 답을 찾을 수 없다면:

            "제공된 문서에서 답을 찾을 수 없습니다."

            라고 답변하세요.

            [문서 Context]
            {context}
            """,
        ),
        (
            "user",
            """
            [질문]
            {question}
            """,
        ),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"question": question, "context": context})

print()
print("result : ", result)

Rewrite:
모바일 금융거래 주의사항은 무엇인가?


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10385.60it/s]



result :  모바일 금융거래를 할 때 주의해야 할 사항은 다음과 같습니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않기
3. 금융거래 비밀번호를 안전하게 관리하기
4. 모바일폰 분실·도난시 모바일폰 금융서비스 사용 중지하기
5. 모바일폰 교체·수리 전 중요 정보 삭제하기
6. 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기
7. 모바일폰 사용환경을 임의로 변경하지 않기
8. 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기
9. 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기
10. 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용시 주의하기

이러한 사항들을 준수하는 것이 보안을 강화하고 사고를 예방하는 데 도움이 됩니다. [출처: p.26]


### RAG Evaluation(평가)

- 필요한 문서를 제대로 가져왔는지 평가


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL, temperature=0)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)

cross_encoder = HuggingFaceCrossEncoder(
    model_name=RERANK_MODEL,
    model_kwargs={"device": "mps"},
)
reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=10,   # ← 아래 설명 참고
)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """
다음 사용자의 질문을 문서 검색에 적합한 형태로
다시 작성하세요.

조건:
- 질문의 원래 의미를 유지하세요.
- 불필요한 표현은 제거하세요.
- 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
- 질문 하나만 출력하세요.
- 설명이나 번호는 출력하지 마세요.

사용자 질문:
{question}
"""
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()


def evaluate_hit_rate(
    question,
    relevant_pages,
    k=5,
    use_rerank=True # true/false로 결과를 찍어서 비교해보면 리랭크가 필요한 지 알 수 있음
):
    rewritten_question = rewrite_chain.invoke({"question": question})

    retrieved_docs = hybrid_retriever.invoke(rewritten_question)

    if use_rerank:
        retrieved_docs = reranker.compress_documents(
            documents=retrieved_docs,
            query=question,
        )

    top_docs = retrieved_docs[:k]

    retrieved_pages = [doc.metadata.get("page") for doc in top_docs]

    hit = any(page in relevant_pages for page in retrieved_pages)

    return hit, retrieved_pages, rewritten_question


evaluation_data = [
    {
        "question": "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?",
        "relevant_pages": [25, 16, 21],
    },
]
hits = []

for item in evaluation_data:
    hit, pages, rewritten_question = evaluate_hit_rate(
        question=item["question"],
        relevant_pages=item["relevant_pages"],
        k=5,
    )

    hits.append(int(hit))

    print("=" * 60)

    print("원래 질문:")
    print(item["question"])

    print("\nRewrite된 질문:")
    print(rewritten_question)

    print("\n정답 페이지:")
    print(item["relevant_pages"])

    print("\n검색된 Top 5 페이지:")
    print(pages)

    print("\nHit@5:")
    print(int(hit))

hit_rate = sum(hits) / len(hits)

print("\n" + "=" * 60)
print(f"최종 Hit Rate@5: {hit_rate:.4f}")
print("=" * 60)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10775.72it/s]


원래 질문:
모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?

Rewrite된 질문:
모바일 금융거래 주의사항은 무엇인가?

정답 페이지:
[25, 16, 21]

검색된 Top 5 페이지:
[25, 24, 36, 142, 16]

Hit@5:
1

최종 Hit Rate@5: 1.0000


### LLM-as-a-Judge

- LLM에게 다른 LLM의 답변을 채점 시키는 것

### Context Relevance

- 검색해서 가져온 Context가 질문에 관련되어 있는지 체크

### Faithfulness

- 환각(Hallucination)을 얼마나 하고 있는지 확인

### Answer Relevance

- 질문에 제대로 답변 했는지 확인


In [ ]:
import os
import re

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL, temperature=0)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
cross_encoder = HuggingFaceCrossEncoder(
    model_name=RERANK_MODEL,
    model_kwargs={"device": "mps"},
)
reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=10,
)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

parser = StrOutputParser()

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로
    다시 작성하세요.
    
    조건:
    - 질문의 원래 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.
    
    사용자 질문:
    {question}
    """
)
rewrite_chain = rewrite_prompt | llm | parser

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
        "system",
        """
        당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.
        
        반드시 아래 제공된 문서 내용만 근거로 답변하세요.
        
        문서에 답이 없다면
        "제공된 문서에서 답을 찾을 수 없습니다."
        라고 답변하세요.
        
        [문서 내용]
        {context}
        """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)
answer_chain = answer_prompt | llm | parser

context_eval_prompt = ChatPromptTemplate.from_template(
    """
    당신은 RAG 검색 품질 평가자입니다.
    
    사용자 질문과 검색된 문서를 보고,
    검색된 문서가 질문에 얼마나 관련 있는지 평가하세요.
    
    점수:
    0.0 = 전혀 관련 없음
    0.5 = 일부 관련
    1.0 = 매우 관련 있음
    
    반드시 숫자 하나만 출력하세요.
    
    [질문]
    {question}
    
    [Context]
    {context}
    """
)
context_eval_chain = context_eval_prompt | llm | parser

faithfulness_prompt = ChatPromptTemplate.from_template(
    """
    당신은 RAG 답변의 사실성을 평가하는 평가자입니다.
    
    답변에 포함된 내용이 제공된 Context에 의해
    뒷받침되는지를 평가하세요.
    
    Context에 없는 내용을 답변이 추가했다면
    점수를 낮게 주세요.
    
    점수:
    0.0 = 대부분 근거 없음
    0.5 = 일부만 근거 있음
    1.0 = 모든 내용이 Context에 근거함
    
    반드시 숫자 하나만 출력하세요.
    
    [Context]
    {context}
    
    [Answer]
    {answer}
    """
)
faithfulness_chain = faithfulness_prompt | llm | parser

answer_relevance_prompt = ChatPromptTemplate.from_template(
    """
    당신은 RAG 답변의 질문 관련성을 평가하는 평가자입니다.
    
    사용자의 질문과 답변을 비교하여
    답변이 질문에 얼마나 적절하게 답했는지 평가하세요.
    
    점수:
    0.0 = 질문에 답하지 못함
    0.5 = 일부만 답함
    1.0 = 질문에 정확하게 답함
    
    반드시 숫자 하나만 출력하세요.
    
    [질문]
    {question}
    
    [Answer]
    {answer}
    """
)
answer_relevance_chain = answer_relevance_prompt | llm | parser

question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"

rewritten_question = rewrite_chain.invoke({"question": question})

print("=" * 60)
print("원래 질문:")
print(question)
print("\nRewrite 질문:")
print(rewritten_question)

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(
    documents=retrieved_docs,
    query=question
)

context = "\n\n".join(doc.page_content for doc in reranked_docs)

answer = answer_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)

print("\n" + "=" * 60)
print("최종 Answer:")
print(answer)

context_score = context_eval_chain.invoke(
    {
        "question": question,
        "context": context,
    }
)

faithfulness_score = faithfulness_chain.invoke(
    {
        "context": context,
        "answer": answer
    }
)

answer_relevance_score = answer_relevance_chain.invoke(
    {
        "question": question,
        "answer": answer
    }
)

def parse_score(text: str) -> float | None:
    """LLM 출력에서 0~1 사이 숫자 하나를 꺼낸다. 실패하면 None"""
    match = re.search(r"\d+(\.\d+)?", text)
    if not match:
        return None
    score = float(match.group())
    return score if 0.0 <= score <= 1.0 else None

print("\n" + "=" * 60)
print("RAG Evaluation")
print("-" * 60)

print("Context Relevance:", parse_score(context_score))
print("Faithfulness:", parse_score(faithfulness_score))
print("Answer Relevance:", parse_score(answer_relevance_score))

print("=" * 60)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 11196.59it/s]


원래 질문:
모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?

Rewrite 질문:
모바일 금융거래 주의사항은 무엇인가?

최종 Answer:
모바일 금융거래를 할 때 주의해야 할 사항은 다음과 같습니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기.
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않기.
3. 금융거래 비밀번호를 안전하게 관리하기.
4. 모바일폰 분실·도난 시 모바일폰 금융서비스 사용 중지하기.
5. 모바일폰 교체·수리 전 중요 정보 삭제하기.
6. 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기.
7. 모바일폰 사용환경을 임의로 변경하지 않기.
8. 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기.
9. 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기.
10. 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용 시 주의하기.

RAG Evaluation
------------------------------------------------------------
Context Relevance: 1.0
Faithfulness: 1.0
Answer Relevance: 1.0
